In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [4]:
#Sample Dataset
english_sentences = [
    "I love you",
    "Hello",
    "How are you",
    "Good morning",
    "Thank you",
    "Good night",
    "I am happy",
    "What is your name",
    "I am fine",
    "See you later",
    "Where are you",
    "I am hungry",
    "I am tired",
    "You are beautiful",
    "I miss you",
]

french_sentences = [
    "Je t'aime",
    "Bonjour",
    "Comment allez-vous",
    "Bonjour matin",
    "Merci",
    "Bonne nuit",
    "Je suis heureux",
    "Quel est votre nom",
    "Je vais bien",
    "À bientôt",
    "Où êtes-vous",
    "J'ai faim",
    "Je suis fatigué",
    "Vous êtes belle",
    "Tu me manques",
]

# Add <sos> / <eos> to French sentences
french_input  = ["<sos> " + s          for s in french_sentences]
french_target = [s + " <eos>"          for s in french_sentences]

print("=" * 55)
print("  Seq2Seq English→French  |  TensorFlow/Keras")
print("=" * 55)
print(f"\nDataset size : {len(english_sentences)} sentence pairs")
print("\nSample pair:")
print(f"  EN : {english_sentences[0]}")
print(f"  FR : {french_sentences[0]}")

  Seq2Seq English→French  |  TensorFlow/Keras

Dataset size : 15 sentence pairs

Sample pair:
  EN : I love you
  FR : Je t'aime


In [5]:
#Tokenisation & Vocabulary

# English tokeniser
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english_sentences)
eng_vocab_size = len(eng_tokenizer.word_index) + 1       # +1 for padding index 0

# French tokeniser (trained on *both* input and target to capture all tokens)
# We use filters="" so that < and > are NOT stripped from <sos>/<eos>
fra_tokenizer = Tokenizer(filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n')
fra_tokenizer.fit_on_texts(french_input + french_target)
fra_vocab_size = len(fra_tokenizer.word_index) + 1

print(f"\nEnglish vocabulary size : {eng_vocab_size}")
print(f"French  vocabulary size : {fra_vocab_size}")

# Integer encode
eng_sequences      = eng_tokenizer.texts_to_sequences(english_sentences)
fra_in_sequences   = fra_tokenizer.texts_to_sequences(french_input)
fra_out_sequences  = fra_tokenizer.texts_to_sequences(french_target)

# Pad to uniform length (post-padding with 0s)
max_eng_len = max(len(s) for s in eng_sequences)
max_fra_len = max(max(len(s) for s in fra_in_sequences),
                  max(len(s) for s in fra_out_sequences))

encoder_input_data = pad_sequences(eng_sequences,     maxlen=max_eng_len, padding="post")
decoder_input_data = pad_sequences(fra_in_sequences,  maxlen=max_fra_len, padding="post")
decoder_target_raw = pad_sequences(fra_out_sequences, maxlen=max_fra_len, padding="post")

print(f"\nMax English length : {max_eng_len}")
print(f"Max French  length : {max_fra_len}")
print(f"\nencoder_input_data shape : {encoder_input_data.shape}")
print(f"decoder_input_data shape : {decoder_input_data.shape}")

# One-hot encode the decoder *target* (shape: samples, timesteps, vocab)
decoder_target_data = tf.keras.utils.to_categorical(decoder_target_raw, num_classes=fra_vocab_size)
print(f"decoder_target_data shape: {decoder_target_data.shape}  (one-hot)")


English vocabulary size : 25
French  vocabulary size : 32

Max English length : 4
Max French  length : 5

encoder_input_data shape : (15, 4)
decoder_input_data shape : (15, 5)
decoder_target_data shape: (15, 5, 32)  (one-hot)


In [6]:
#Model Hyperparameters

EMBEDDING_DIM = 64    # word embedding dimensionality
LSTM_UNITS    = 128   # number of hidden units in each LSTM
BATCH_SIZE    = 4
EPOCHS        = 200

In [7]:
#Build the Training Model
#Encoder
encoder_inputs = Input(shape=(max_eng_len,), name="encoder_input")
enc_emb        = Embedding(eng_vocab_size, EMBEDDING_DIM,
                            mask_zero=True, name="encoder_embedding")(encoder_inputs)
_, enc_h, enc_c = LSTM(LSTM_UNITS, return_state=True, name="encoder_lstm")(enc_emb)
encoder_states = [enc_h, enc_c]
#Decoder
decoder_inputs = Input(shape=(max_fra_len,), name="decoder_input")
dec_emb_layer  = Embedding(fra_vocab_size, EMBEDDING_DIM,
                             mask_zero=True, name="decoder_embedding")
dec_emb        = dec_emb_layer(decoder_inputs)

decoder_lstm   = LSTM(LSTM_UNITS, return_sequences=True,
                       return_state=True, name="decoder_lstm")
decoder_out, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense  = Dense(fra_vocab_size, activation="softmax", name="output_dense")
decoder_output = decoder_dense(decoder_out)

# Combine into one trainable model
model = Model([encoder_inputs, decoder_inputs], decoder_output)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

print("\n" + "─" * 55)
print("  MODEL SUMMARY")
print("─" * 55)
model.summary()


───────────────────────────────────────────────────────
  MODEL SUMMARY
───────────────────────────────────────────────────────


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 4, 64)     │      1,600 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_6         │ (None, 4)         │          0 │ encoder_input[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 5, 64)     │      2,048 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 128),     │     98,816 │ encoder_embeddin… │
│                     │ (None, 128),      │            │ not_equal_6[0][0] │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 5, 128),  │     98,816 │ decoder_embeddin… │
│                     │ (None, 128),      │            │ encoder_lstm[0][… │
│                     │ (None, 128)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_dense        │ (None, 5, 32)     │      4,128 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 205,408 (802.38 KB)

 Trainable params: 205,408 (802.38 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# Train

print("\n" + "─" * 55)
print("  TRAINING")
print("─" * 55)

history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    verbose=0,
    callbacks=[
        tf.keras.callbacks.LambdaCallback(
            on_epoch_end=lambda epoch, logs:
                print(f"  Epoch {epoch+1:>3}/{EPOCHS}  "
                      f"loss={logs['loss']:.4f}  "
                      f"acc={logs['accuracy']:.4f}")
                if (epoch + 1) % 50 == 0 else None
        )
    ]
)

print("\nTraining complete.")


───────────────────────────────────────────────────────
  TRAINING
───────────────────────────────────────────────────────
  Epoch  50/200  loss=0.4423  acc=0.9773
  Epoch 100/200  loss=0.0363  acc=1.0000
  Epoch 150/200  loss=0.0121  acc=1.0000
  Epoch 200/200  loss=0.0063  acc=1.0000

Training complete.


In [9]:
#Build Inference Models

# Encoder inference model: English → (h, c)
encoder_model = Model(encoder_inputs, encoder_states, name="encoder_inference")

# Decoder inference model: takes (token, h, c) → (next_token, h', c')
dec_state_h_in = Input(shape=(LSTM_UNITS,), name="dec_h_input")
dec_state_c_in = Input(shape=(LSTM_UNITS,), name="dec_c_input")

# Decoder input is a SINGLE token at each step → shape (batch, 1)
dec_inf_input  = Input(shape=(1,), name="dec_token_input")
dec_inf_emb    = dec_emb_layer(dec_inf_input)

dec_inf_out, dec_h_out, dec_c_out = decoder_lstm(
    dec_inf_emb,
    initial_state=[dec_state_h_in, dec_state_c_in]
)
dec_inf_dense = decoder_dense(dec_inf_out)

decoder_model = Model(
    [dec_inf_input, dec_state_h_in, dec_state_c_in],
    [dec_inf_dense, dec_h_out, dec_c_out],
    name="decoder_inference"
)

print("\nInference models built successfully.")



Inference models built successfully.


In [10]:
# Inference Helper

fra_index_word = {v: k for k, v in fra_tokenizer.word_index.items()}

def translate(input_sentence: str, max_output_len: int = 20):

    seq = eng_tokenizer.texts_to_sequences([input_sentence])
    padded = pad_sequences(seq, maxlen=max_eng_len, padding="post")

    states = encoder_model.predict(padded, verbose=0)

    sos_index = fra_tokenizer.word_index["<sos>"]
    eos_index = fra_tokenizer.word_index["<eos>"]
    target_token = np.array([[sos_index]])

    translated_words = []
    h, c = states[0], states[1]

    for _ in range(max_output_len):
        probs, h, c = decoder_model.predict([target_token, h, c], verbose=0)

        token_idx = np.argmax(probs[0, -1, :])

        if token_idx == eos_index:
            break

        word = fra_index_word.get(token_idx, "")
        if word:
            translated_words.append(word)

        target_token = np.array([[token_idx]])

    return " ".join(translated_words)

In [11]:
#Test the Model
print("\n" + "=" * 55)
print("  INFERENCE – Translation Results")
print("=" * 55)

test_sentences = [
    "I love you",
    "Hello",
    "Good morning",
    "Thank you",
    "I am happy",
    "Good night",
]

print(f"\n{'English':<25} {'Predicted French':<30} {'Expected French'}")
print("─" * 75)
for en, fr in zip(test_sentences, french_sentences[:len(test_sentences)]):
    predicted = translate(en)
    print(f"{en:<25} {predicted:<30} {fr}")

print("\n" + "─" * 55)
print("  CUSTOM INPUT TEST")
print("─" * 55)
custom = "I miss you"
print(f"  Input    : {custom}")
print(f"  Output   : {translate(custom)}")
print(f"  Expected : {french_sentences[english_sentences.index(custom)]}")


  INFERENCE – Translation Results

English                   Predicted French               Expected French
───────────────────────────────────────────────────────────────────────────
I love you                je t'aime                      Je t'aime
Hello                     bonjour                        Bonjour
Good morning              bonjour matin                  Comment allez-vous
Thank you                 merci                          Bonjour matin
I am happy                je suis heureux                Merci
Good night                bonne nuit                     Bonne nuit

───────────────────────────────────────────────────────
  CUSTOM INPUT TEST
───────────────────────────────────────────────────────
  Input    : I miss you
  Output   : je t'aime
  Expected : Tu me manques
